In [1]:
import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# =========================
# 1. 경로 설정
# =========================
raw_path = "/content/raw_data_katchers.csv"   # 파일 업로드한 경로에 맞게 수정
save_dir = "/content/katchers_split"
os.makedirs(save_dir, exist_ok=True)

# =========================
# 2. 원본 데이터 불러오기
# =========================
data = pd.read_csv(raw_path, sep="\t", encoding="utf-8", low_memory=False)

print("원본 데이터:", data.shape)
print(data.columns.tolist())

# =========================
# 3. 교수님 코드와 동일한 전처리
# =========================

# Step 1: 구매일이 2일 이상 있는 유저만 사용
data["unique_purchase_dates"] = data.groupby("user_id")["initial_paid_at"].transform(lambda x: x.nunique())
data = data[data["unique_purchase_dates"] > 1].drop(columns=["unique_purchase_dates"])

# Step 2: 유저별 가장 최근 구매일 찾기
df_grouped = data.groupby("user_id")["initial_paid_at"].max().reset_index(name="initial_paid_at")

# Step 3: 가장 최근 구매일에 구매한 모든 상품 추출
df_max_all = data.merge(df_grouped, how="inner", on=["user_id", "initial_paid_at"])

# Step 4: 최근 구매일 상품 중 유저별 정답 상품 1개 랜덤 선택
df_max = df_max_all.groupby("user_id").sample(n=1, random_state=42).reset_index(drop=True)

# Step 5: 마지막 구매일이 아닌 구매 이력 추출
df_joined = data.merge(df_grouped, how="inner", on="user_id", suffixes=["", "_y"])
df_joined["initial_paid_at_mismatch"] = df_joined["initial_paid_at"] != df_joined["initial_paid_at_y"]

df_not_max = df_joined[df_joined["initial_paid_at_mismatch"]].drop(
    columns=["initial_paid_at_y", "initial_paid_at_mismatch"]
)

# Step 6: 정답 상품 1개 + 이전 구매 이력 합치기
df_processed = pd.concat([df_max, df_not_max]).reset_index(drop=True)

# Step 7: 구매 상품 수 기준 필터링
df_processed["total_order_cnt"] = df_processed.groupby("user_id")["product_id"].transform("count")
df_processed = df_processed[
    (df_processed["total_order_cnt"] > 1) &
    (df_processed["total_order_cnt"] < 68)
].drop(columns=["total_order_cnt"]).reset_index(drop=True)

print("전처리 후 데이터:", df_processed.shape)
print("전처리 후 유저 수:", df_processed["user_id"].nunique())
print("전처리 후 상품 수:", df_processed["product_id"].nunique())

# =========================
# 4. 교수님 코드와 동일한 user_id 기준 split
# =========================

# 1차 분리: train_val 80%, test 20%
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_val_idx, test_idx = next(
    gss.split(df_processed, groups=df_processed["user_id"])
)

train_val_data = df_processed.iloc[train_val_idx].reset_index(drop=True)
test_data = df_processed.iloc[test_idx].reset_index(drop=True)

# 2차 분리: train_val 중 train 75%, val 25%
# 최종 비율: train 60%, val 20%, test 20%
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=42)
train_idx, val_idx = next(
    gss.split(train_val_data, groups=train_val_data["user_id"])
)

train_data = train_val_data.iloc[train_idx].reset_index(drop=True)
val_data = train_val_data.iloc[val_idx].reset_index(drop=True)

# =========================
# 5. 저장
# =========================
train_data.to_csv(os.path.join(save_dir, "train_data.csv"), index=False, encoding="utf-8-sig")
val_data.to_csv(os.path.join(save_dir, "val_data.csv"), index=False, encoding="utf-8-sig")
test_data.to_csv(os.path.join(save_dir, "test_data.csv"), index=False, encoding="utf-8-sig")

# =========================
# 6. 확인 출력
# =========================
print("\n저장 완료")
print("train:", train_data.shape, "유저 수:", train_data["user_id"].nunique())
print("val  :", val_data.shape, "유저 수:", val_data["user_id"].nunique())
print("test :", test_data.shape, "유저 수:", test_data["user_id"].nunique())

# 유저 겹침 확인
train_users = set(train_data["user_id"])
val_users = set(val_data["user_id"])
test_users = set(test_data["user_id"])

print("\n유저 겹침 확인")
print("train ∩ val :", len(train_users & val_users))
print("train ∩ test:", len(train_users & test_users))
print("val ∩ test  :", len(val_users & test_users))

print("\n저장 경로:", save_dir)

원본 데이터: (165318, 28)
['user_id', 'code', 'order_id', 'order_code', 'option_product_code', 'option_product_id', 'full_name', 'status_code', 'status_changed_at', 'seller_product_id', 'product_id', 'product_name', 'product_sales_type', 'category_id', 'category_name', 'root_category_id', 'root_category_name', 'original_price', 'unit_price', 'quantity', 'total_product_price', 'initial_paid_method', 'initial_paid_amount', 'initial_paid_at', 'attribute_types', 'attribute_values', 'attributes', 'day']
전처리 후 데이터: (107001, 28)
전처리 후 유저 수: 16787
전처리 후 상품 수: 1608

저장 완료
train: (64676, 28) 유저 수: 10071
val  : (21008, 28) 유저 수: 3358
test : (21317, 28) 유저 수: 3358

유저 겹침 확인
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

저장 경로: /content/katchers_split
